In [1]:
import warnings
import numpy as np
import pandas as pd
import features as fe

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
SEED = 42
np.random.seed(SEED)

In [2]:
train, test, events = fe.load_data("data")
print("train", train.shape, "test", test.shape, "events", events.shape)
print("bot share:", round(train.target.mean(), 4))
print("train windows:", train.window_start_ts.min().date(), "-", train.window_start_ts.max().date())
print("test windows:", test.window_start_ts.min().date(), "-", test.window_start_ts.max().date())
print("window length:", (train.window_end_ts - train.window_start_ts).unique())
print("train/test cookie overlap:", len(set(train.cookie_id) & set(test.cookie_id)))

train (11091, 5) test (4909, 4) events (328905, 14)
bot share: 0.0811
train windows: 2026-04-06 - 2026-04-19
test windows: 2026-04-20 - 2026-04-26
window length: <TimedeltaArray>
['1 days']
Length: 1, dtype: timedelta64[ns]
train/test cookie overlap: 0


In [3]:
print("duplicate rows:", events.duplicated().sum())
print("sorted by time:", events.event_ts.is_monotonic_increasing)
print(events.platform.value_counts().to_dict())
events.isna().mean().round(3).to_frame("nan_share").T

duplicate rows: 4863
sorted by time: False


{'desktop': 46866, 'WEB': 46476, 'Web': 46365, 'web': 45951, 'ANDROID': 42462, 'Android': 42254, 'android': 42159, 'iphone': 4103, 'IOS': 4100, 'iOS': 4091, 'ios': 4078}


,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
nan_share,0.0,0.0,0.0,0.0,0.0,0.0,0.348,0.1,0.072,0.407,0.695,0.695,0.67,0.67


In [4]:
meta_all = pd.concat([train, test])
m = events.merge(meta_all[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id")
m["position"] = np.select([m.event_ts < m.window_start_ts, m.event_ts >= m.window_end_ts],
                          ["before", "after"], "inside")
m["split"] = np.where(m.cookie_id.isin(train.cookie_id), "train", "test")
display(pd.crosstab(m.event_name, m.position))
after = m[m.position == "after"].groupby("cookie_id").size()
print("events after window per cookie (train):",
      train.assign(n=train.cookie_id.map(after).fillna(0)).groupby("target").n.mean().round(2).to_dict())
print("events after window in test:", (m[m.split == "test"].position == "after").sum())

position,after,inside
event_name,,
captcha_shown,7928,0
contact_chat_open,577,5548
contact_message_sent,258,2823
contact_phone_show,1028,10288
favorite_add,1753,17296
item_view,12731,108086
login,606,5661
photo_swipe,3433,33084
search_results_view,10604,89798


events after window per cookie (train): {0: 2.08, 1: 21.77}
events after window in test: 0


In [5]:
inside = m[(m.position == "inside") & (m.split == "train")].merge(train[["cookie_id", "target"]], on="cookie_id")
display(pd.crosstab(inside.event_name, inside.target, normalize="columns").round(3))
dup = events.duplicated(keep=False)
print("duplicate share by target:",
      events.assign(d=dup).merge(train, on="cookie_id").groupby("target").d.mean().round(4).to_dict())

target,0,1
event_name,,
contact_chat_open,0.020,0.014
contact_message_sent,0.010,0.008
contact_phone_show,0.037,0.026
favorite_add,0.066,0.027
item_view,0.365,0.436
login,0.021,0.008
photo_swipe,0.123,0.070
search_results_view,0.305,0.352
seller_page_view,0.053,0.059


duplicate share by target: {0: 0.0297, 1: 0.0285}


In [6]:
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from metric import precision_at_recall

ev = fe.clean_events(events)
y = train.target.values
day = train.window_start_ts
FOLDS = [("2026-04-12", "2026-04-14"), ("2026-04-14", "2026-04-16"),
         ("2026-04-16", "2026-04-18"), ("2026-04-18", "2026-04-20")]
LGB_PARAMS = dict(n_estimators=500, learning_rate=0.03, num_leaves=15, min_child_samples=50,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0)


def lgb_model(seed=SEED):
    return lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed, deterministic=True, verbose=-1)


def fit_predict(kind, X_tr, y_tr, X_va, seeds=(SEED,)):
    if kind == "lgb":
        return np.mean([lgb_model(s).fit(X_tr, y_tr).predict_proba(X_va)[:, 1] for s in seeds], axis=0)
    X_all = pd.get_dummies(pd.concat([X_tr, X_va]), dtype=float)
    X_tr, X_va = X_all.iloc[:len(X_tr)], X_all.iloc[len(X_tr):]
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LogisticRegression(C=0.3, max_iter=2000))
    return model.fit(X_tr, y_tr).predict_proba(X_va)[:, 1]


def time_cv(X, cols, kind="lgb", seeds=(SEED,)):
    rows = []
    for start, end in FOLDS:
        tr = (day < start).values
        va = ((day >= start) & (day < end)).values
        p = fit_predict(kind, X.loc[tr, cols], y[tr], X.loc[va, cols], seeds)
        rows.append({"P@R0.7": precision_at_recall(y[va], p),
                     "PR-AUC": average_precision_score(y[va], p),
                     "ROC-AUC": roc_auc_score(y[va], p)})
    r = pd.DataFrame(rows)
    return {"P@R0.7": r["P@R0.7"].mean(), "std": r["P@R0.7"].std(),
            "PR-AUC": r["PR-AUC"].mean(), "ROC-AUC": r["ROC-AUC"].mean(),
            "folds": np.round(r["P@R0.7"].values, 3)}

In [7]:
X_basic = fe.basic_features(ev, train)
results = {
    "const": {"P@R0.7": np.mean([y[((day >= a) & (day < b)).values].mean() for a, b in FOLDS])},
    "baseline: lgb, n_events + item_nunique": time_cv(X_basic, ["n_events", "item_nunique"]),
}
pd.DataFrame(results).T.round(4)

,P@R0.7,std,PR-AUC,ROC-AUC,folds
const,0.079478,NaN,NaN,NaN,NaN
"baseline: lgb, n_events + item_nunique",0.107734,0.013836,0.222329,0.656702,"[0.088, 0.109, 0.119, 0.115]"


In [8]:
popularity = fe.item_popularity_features(ev, meta_all, lookback_days=7)
X_train = fe.build_features(ev, train, with_ua=True, popularity=popularity)
X_test = fe.build_features(ev, test, with_ua=True, popularity=popularity)
assert (X_train.cookie_id.values == train.cookie_id.values).all()
assert (X_test.cookie_id.values == test.cookie_id.values).all()
UA_COLS = ["ua_nunique", "ua_automation", "ua_chrome_ver", "ua_is_app"]
BASE_COLS = [c for c in X_train.columns if c not in ["cookie_id"] + UA_COLS and not c.startswith("item_pop")]
print(X_train.shape, X_test.shape, len(BASE_COLS))

(11091, 68) (4909, 68) 60


In [9]:
results.update({
    "logreg, behavioral": time_cv(X_train, BASE_COLS, kind="logreg"),
    "lgb, behavioral": time_cv(X_train, BASE_COLS),
    "lgb, behavioral + UA": time_cv(X_train, BASE_COLS + UA_COLS),
})
pd.DataFrame(results).T.round(4)

,P@R0.7,std,PR-AUC,ROC-AUC,folds
const,0.079478,NaN,NaN,NaN,NaN
"baseline: lgb, n_events + item_nunique",0.107734,0.013836,0.222329,0.656702,"[0.088, 0.109, 0.119, 0.115]"
"logreg, behavioral",0.58021,0.072206,0.693584,0.917459,"[0.535, 0.653, 0.629, 0.504]"
"lgb, behavioral",0.768071,0.033406,0.783386,0.936084,"[0.743, 0.736, 0.788, 0.805]"
"lgb, behavioral + UA",0.748421,0.025194,0.783257,0.935651,"[0.712, 0.754, 0.769, 0.759]"


In [10]:
print(X_train.groupby(day.dt.date)[["item_pop_mean", "item_pop_mean_rel"]].mean().round(3).T)
POP_RAW = ["item_pop_mean", "item_pop_zero_share"]
POP_COLS = ["item_pop_mean_rel", "item_pop_zero_share"]
FINAL_COLS = BASE_COLS + POP_COLS
results.update({
    "lgb, + popularity raw": time_cv(X_train, BASE_COLS + POP_RAW),
    "lgb, + popularity raw + rel": time_cv(X_train, BASE_COLS + POP_RAW + ["item_pop_mean_rel"]),
    "lgb, + popularity rel": time_cv(X_train, FINAL_COLS),
    "lgb, + popularity rel + UA": time_cv(X_train, FINAL_COLS + UA_COLS),
})
pd.DataFrame(results).T.round(4)

window_start_ts    2026-04-06  2026-04-07  2026-04-08  2026-04-09  2026-04-10  2026-04-11  2026-04-12  2026-04-13  2026-04-14  2026-04-15  2026-04-16  2026-04-17  2026-04-18  2026-04-19
item_pop_mean            0.13        0.29       0.431       0.579       0.752       0.916       1.035       1.103       1.136       1.092       1.068       1.027       1.006       0.933
item_pop_mean_rel        1.00        1.00       1.000       1.000       1.000       1.000       1.000       1.000       1.000       1.000       1.000       1.000       1.000       1.000


,P@R0.7,std,PR-AUC,ROC-AUC,folds
const,0.079478,NaN,NaN,NaN,NaN
"baseline: lgb, n_events + item_nunique",0.107734,0.013836,0.222329,0.656702,"[0.088, 0.109, 0.119, 0.115]"
"logreg, behavioral",0.58021,0.072206,0.693584,0.917459,"[0.535, 0.653, 0.629, 0.504]"
"lgb, behavioral",0.768071,0.033406,0.783386,0.936084,"[0.743, 0.736, 0.788, 0.805]"
"lgb, behavioral + UA",0.748421,0.025194,0.783257,0.935651,"[0.712, 0.754, 0.769, 0.759]"
"lgb, + popularity raw",0.788443,0.054908,0.800506,0.941755,"[0.717, 0.782, 0.848, 0.807]"
"lgb, + popularity raw + rel",0.837476,0.047613,0.803764,0.94183,"[0.867, 0.826, 0.881, 0.775]"
"lgb, + popularity rel",0.856697,0.051749,0.805781,0.941732,"[0.903, 0.841, 0.892, 0.791]"
"lgb, + popularity rel + UA",0.832531,0.057402,0.807301,0.943195,"[0.894, 0.833, 0.848, 0.756]"


In [11]:
from sklearn.model_selection import StratifiedKFold


def adversarial_auc(cols):
    A = pd.concat([X_train[cols], X_test[cols]], ignore_index=True)
    ya = np.r_[np.zeros(len(X_train)), np.ones(len(X_test))]
    pa = np.zeros(len(A))
    for tr_idx, va_idx in StratifiedKFold(5, shuffle=True, random_state=SEED).split(A, ya):
        pa[va_idx] = lgb_model().fit(A.iloc[tr_idx], ya[tr_idx]).predict_proba(A.iloc[va_idx])[:, 1]
    return roc_auc_score(ya, pa)


pd.Series({
    "behavioral": adversarial_auc(BASE_COLS),
    "behavioral + popularity rel": adversarial_auc(FINAL_COLS),
    "behavioral + popularity raw + rel": adversarial_auc(BASE_COLS + POP_RAW + ["item_pop_mean_rel"]),
}).round(3).to_frame("adversarial ROC-AUC")

,adversarial ROC-AUC
behavioral,0.503
behavioral + popularity rel,0.754
behavioral + popularity raw + rel,0.944


In [12]:
BLOCKS = {
    "timing": [c for c in BASE_COLS if c.startswith(("dt_", "duration", "n_sessions", "hours_active",
                                                     "night_share", "first_hour", "max_events"))],
    "pointer": [c for c in BASE_COLS if c.startswith("pointer")],
    "diversity": [c for c in BASE_COLS if c.startswith(("item_", "cat_", "loc_", "query", "n_query",
                                                        "n_item_events", "share_pro", "top_cat", "page_"))],
    "activity": [c for c in BASE_COLS if c.startswith(("share_", "swipes", "contacts", "favs", "views_per",
                                                       "n_contacts", "has_login", "n_event_types"))
                 and c != "share_pro_seller"],
    "meta": ["cookie_age_h", "age_at_first_event_h", "created_in_window"],
    "popularity": POP_COLS,
}
ablation = {"all": time_cv(X_train, FINAL_COLS)}
for block, cols in BLOCKS.items():
    ablation[f"without {block} ({len(cols)})"] = time_cv(X_train, [c for c in FINAL_COLS if c not in cols])
pd.DataFrame(ablation).T[["P@R0.7", "std", "PR-AUC", "folds"]].round(4)

,P@R0.7,std,PR-AUC,folds
all,0.856697,0.051749,0.805781,"[0.903, 0.841, 0.892, 0.791]"
without timing (15),0.747455,0.095055,0.755296,"[0.724, 0.819, 0.824, 0.623]"
without pointer (6),0.677474,0.126207,0.750265,"[0.737, 0.785, 0.497, 0.691]"
without diversity (16),0.781495,0.052869,0.78076,"[0.832, 0.716, 0.817, 0.761]"
without activity (16),0.830615,0.050086,0.793835,"[0.832, 0.833, 0.89, 0.767]"
without meta (3),0.831678,0.039432,0.803583,"[0.85, 0.848, 0.856, 0.773]"
without popularity (2),0.768071,0.033406,0.783386,"[0.743, 0.736, 0.788, 0.805]"


In [13]:
FINAL_SEEDS = (42, 43, 44, 45, 46)
final_cv = time_cv(X_train, FINAL_COLS, seeds=FINAL_SEEDS)
final_cv

{'P@R0.7': np.float64(0.8325283123276462),
 'std': np.float64(0.045017917394395986),
 'PR-AUC': np.float64(0.808998788520769),
 'ROC-AUC': np.float64(0.9428673924568237),
 'folds': array([0.875, 0.812, 0.864, 0.779])}

In [14]:
final_models = [lgb_model(s).fit(X_train[FINAL_COLS], y) for s in FINAL_SEEDS]
imp = pd.Series(np.mean([m.booster_.feature_importance("gain") for m in final_models], axis=0), FINAL_COLS)
(imp / imp.sum() * 100).sort_values(ascending=False).head(15).round(2).to_frame("gain, %")

,"gain, %"
dt_median,14.53
item_pop_mean_rel,11.46
pointer_x_std,8.99
loc_nunique,5.54
pointer_edge_share,5.33
cat_per_event,3.97
dt_min,3.15
page_mean,2.77
cat_nunique,2.19
item_repeat_ratio,1.98


In [15]:
score = np.mean([m.predict_proba(X_test[FINAL_COLS])[:, 1] for m in final_models], axis=0)
submission = pd.DataFrame({"cookie_id": X_test.cookie_id, "score": score})
assert len(submission) == len(test)
assert submission.cookie_id.is_unique and set(submission.cookie_id) == set(test.cookie_id)
assert submission.score.between(0, 1).all() and submission.score.notna().all()
print("unique scores:", submission.score.nunique(), "| mean score:", round(score.mean(), 4))
submission.to_csv("submission.csv", index=False)
submission.head()

unique scores: 4909 | mean score: 0.0831


,cookie_id,score
0,ck_315fb710a0e371e7,0.000937
1,ck_a76ee3b3e3e522fd,0.037400
2,ck_94c9a4d382689e82,0.038978
3,ck_8eaf9509ad9462a0,0.001351
4,ck_9a88a5a989cb5bc6,0.009190
